In [ ]:
import kagglehub
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [ ]:
# Download latest version
path = kagglehub.competition_download('itobos-2024-detection')

print("Path to competition files:", path)

100%|██████████| 17.8G/17.8G [04:32<00:00, 70.4MB/s]


Extracting files...
Path to competition files: /root/.cache/kagglehub/competitions/itobos-2024-detection


In [ ]:
import os
import shutil
from sklearn.model_selection import train_test_split

# Correct paths
base = "/root/.cache/kagglehub/competitions/itobos-2024-detection"
train_images = f"{base}/_train/_train/images/"
train_labels = f"{base}/_train/_train/labels/"

# Get all images
images = [f for f in os.listdir(train_images) if f.endswith('.png')]
train_imgs, val_imgs = train_test_split(images, test_size=0.2, random_state=42)

# Create YOLO folder structure
for split in ['train', 'val']:
    os.makedirs(f"dataset/images/{split}", exist_ok=True)
    os.makedirs(f"dataset/labels/{split}", exist_ok=True)

# Copy train files
for img in train_imgs:
    shutil.copy(f"{train_images}{img}", f"dataset/images/train/{img}")
    label = img.replace('.png', '.txt')
    shutil.copy(f"{train_labels}{label}", f"dataset/labels/train/{label}")
# Copy val files
for img in val_imgs:
    shutil.copy(f"{train_images}{img}", f"dataset/images/val/{img}")
    label = img.replace('.png', '.txt')
    shutil.copy(f"{train_labels}{label}", f"dataset/labels/val/{label}")

print(f"Train: {len(train_imgs)} images")
print(f"Val: {len(val_imgs)} images")

Train: 6778 images
Val: 1695 images


In [ ]:
yaml_content = """
path: /content/dataset
train: images/train
val: images/val
nc: 1
names:
  0: lesion
"""
with open('dataset.yaml', 'w') as f:
    f.write(yaml_content)

In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.6 MB/s eta 0:00:00


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/Itobos_experiment/lesion_detection/itobos_v11/weights/last.pt")

In [ ]:
# trained for 30 epochs so far mAP50 of .64
results = model.train(
    data = "/content/dataset.yaml",
    imgsz=640,               # Reduced image size to see if it would work/ so it could converge faster
    epochs=15,
    batch=8,
    single_cls=True,
    warmup_epochs=0, # changed to zero since im training the model again
    optimizer="AdamW",
    lr0=0.0005,
    lrf=0.001,
    workers=8,
    project="/content/drive/MyDrive/Itobos_experiment/lesion_detection",
    name="itobos_v11_pt5",
    exist_ok=True,
    patience=7,
    plots=True,              # Generate training plots
    device='0',              # Use GPU 0
    val=True,                # Perform validation
    augment=True  # augmentations to prevent overfitting
)

Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.001, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/Itobos_experiment/lesion_detection/itobos_v11/weights/last.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=itobos_v11_pt5, nbs=64, nms=False, opse